# 01 · Extract & Clean FormFactor Financials from SEC EDGAR

**Goal:** build a clean, analysis-ready dataset of FormFactor's (NASDAQ: FORM) annual financial statements directly from the SEC's XBRL data.

| Step | What happens |
|---|---|
| 1. Extract | Pull every XBRL fact FormFactor has filed via the SEC `companyfacts` API (cached locally) |
| 2. Flatten | Convert the nested JSON into a long table, one row per reported value |
| 3. Clean | Keep annual 10-K values, align the 52/53-week fiscal year, handle restatements and duplicates |
| 4. Map | Translate XBRL tags into 28 standard metrics, with fallback tags |
| 5. Validate | Check accounting identities and reconcile against reported revenue |
| 6. Export | Write long and wide CSVs for the SQL analysis in notebook 02 |

## Setup

In [1]:
import json
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# The SEC requires a User-Agent with contact details on every request (replace with your own)
HEADERS = {"User-Agent": "Chen-Chih Hung cosby432@gmail.com"}

CIK = "1039399"            # FormFactor's SEC Central Index Key
CIK10 = CIK.zfill(10)      # the API expects 10 digits

BASE = Path.cwd()
RAW_DIR = BASE / "data" / "raw"
CLEAN_DIR = BASE / "data" / "clean"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

## 1. Extract

The `companyfacts` endpoint returns every XBRL value a company has ever filed. The raw response is cached in `data/raw/` so the pipeline can be re-run without calling the API again, and so the untouched source is always available if a cleaning step needs to be revisited.

In [2]:
raw_path = RAW_DIR / "formfactor_companyfacts.json"

if raw_path.exists():
    facts = json.loads(raw_path.read_text(encoding="utf-8"))
    print("Loaded cached response")
else:
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK10}.json"
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    facts = resp.json()
    raw_path.write_text(json.dumps(facts), encoding="utf-8")
    print("Downloaded from SEC")

print("Entity:", facts["entityName"])
print("us-gaap concepts:", len(facts["facts"]["us-gaap"]))

Loaded cached response
Entity: FormFactor, Inc.
us-gaap concepts: 520


### 1.1 Structure of the response

```
facts
 └─ "us-gaap"
     └─ concept (e.g. "NetIncomeLoss")
          └─ "units" → "USD" → [ {start, end, val, fy, fp, form, filed, frame, accn}, ... ]
```

Two details drive the cleaning logic below:
- `start`/`end` define the period. Income-statement and cash-flow items have both (a duration); balance-sheet items only have `end` (a point in time).
- `fy` is the fiscal year **of the filing**, not of the value. A FY2025 10-K also reports FY2024 and FY2023 comparatives, so the same value appears in several filings.

In [3]:
ni = facts["facts"]["us-gaap"]["NetIncomeLoss"]["units"]["USD"]
pd.DataFrame(ni).tail(3)

,start,end,val,accn,fy,fp,form,filed,frame
252,2025-12-28,2026-03-28,20384000,0001039399-26-000023,2026,Q1,10-Q,2026-05-05,CY2026Q1
253,2025-12-28,2026-06-27,76591000,0001039399-26-000033,2026,Q2,10-Q,2026-08-04,NaN
254,2026-03-29,2026-06-27,56207000,0001039399-26-000033,2026,Q2,10-Q,2026-08-04,CY2026Q2


## 2. Flatten nested JSON into a long table

In [4]:
rows = []
for taxonomy, concepts in facts["facts"].items():
    for concept, info in concepts.items():
        for unit, entries in info["units"].items():
            for e in entries:
                rows.append({"taxonomy": taxonomy, "concept": concept,
                             "label": info.get("label"), "unit": unit, **e})

df_raw = pd.DataFrame(rows)
df_raw.to_csv(RAW_DIR / "formfactor_facts_long_raw.csv", index=False)
print(df_raw.shape)
df_raw["form"].value_counts().head()

(24086, 13)


form
10-Q    14841
10-K     9241
S-8         4
Name: count, dtype: int64

## 3. Clean

Issues to resolve:
1. Quarterly (10-Q) and annual (10-K) values are mixed → keep 10-K only
2. Some 10-Ks include quarterly durations → keep ~12-month durations only
3. Each value is repeated across consecutive filings → de-duplicate
4. Restated values differ between filings → keep the **most recently filed** version
5. FormFactor uses a 52/53-week fiscal year ending on the last Saturday of December → derive the fiscal year from the period end date, not the `fy` field

In [5]:
df = df_raw[(df_raw["taxonomy"] == "us-gaap") & (df_raw["form"].isin(["10-K", "10-K/A"]))].copy()
for col in ["start", "end", "filed"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["period_type"] = df["start"].isna().map({True: "instant", False: "flow"})
df["duration_days"] = (df["end"] - df["start"]).dt.days

# Duration distribution: ~364/371 days = annual (52/53-week years), ~91 days = quarterly
df.loc[df["period_type"] == "flow", "duration_days"].value_counts().head()

duration_days
363.00    3816
90.00     1122
370.00     959
97.00       55
1.00         4
Name: count, dtype: int64

In [6]:
is_annual_flow = (df["period_type"] == "flow") & df["duration_days"].between(350, 380)
df = df[is_annual_flow | (df["period_type"] == "instant")].copy()

# Fiscal year from the period end date (periods ending Jan-May belong to the prior year)
df["fiscal_year"] = df["end"].dt.year.where(df["end"].dt.month >= 6, df["end"].dt.year - 1)

# Balance-sheet values: keep only fiscal year-end dates (taken from annual revenue periods)
fy_end_dates = set(df.loc[(df["period_type"] == "flow")
                          & (df["concept"] == "RevenueFromContractWithCustomerExcludingAssessedTax"), "end"])
df = df[(df["period_type"] == "flow") | df["end"].isin(fy_end_dates)].copy()
print("Fiscal year-ends:", sorted(d.date() for d in fy_end_dates)[-5:])

Fiscal year-ends: [datetime.date(2021, 12, 25), datetime.date(2022, 12, 31), datetime.date(2023, 12, 30), datetime.date(2024, 12, 28), datetime.date(2025, 12, 27)]


In [7]:
key = ["concept", "unit", "fiscal_year", "period_type"]

# How many (concept, year) pairs were reported with different values in different filings?
n_versions = df.groupby(key)["val"].nunique()
print(f"{(n_versions > 1).sum()} concept-years were restated or reclassified across filings")

# Keep the most recently filed value
df = (df.sort_values("filed")
        .drop_duplicates(subset=key, keep="last")
        .reset_index(drop=True))
print("Rows after de-duplication:", len(df))

124 concept-years were restated or reclassified across filings
Rows after de-duplication: 3316


## 4. Map XBRL tags to standard metrics

The same line item can be tagged differently across companies or years (e.g. `CostOfRevenue` vs `CostOfGoodsAndServicesSold`). Each metric therefore has an ordered list of candidate tags, and the first tag with data is used **year by year**.

In [8]:
METRICS = {
    # Income statement
    "revenue":            ("IS", ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues", "SalesRevenueNet"]),
    "cost_of_revenue":    ("IS", ["CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfGoodsSold"]),
    "gross_profit":       ("IS", ["GrossProfit"]),
    "rd_expense":         ("IS", ["ResearchAndDevelopmentExpense"]),
    "sga_expense":        ("IS", ["SellingGeneralAndAdministrativeExpense"]),
    "operating_income":   ("IS", ["OperatingIncomeLoss"]),
    "income_tax":         ("IS", ["IncomeTaxExpenseBenefit"]),
    "net_income":         ("IS", ["NetIncomeLoss"]),
    "eps_diluted":        ("IS", ["EarningsPerShareDiluted"]),
    "shares_diluted":     ("IS", ["WeightedAverageNumberOfDilutedSharesOutstanding"]),
    # Balance sheet
    "cash":               ("BS", ["CashAndCashEquivalentsAtCarryingValue"]),
    "st_investments":     ("BS", ["MarketableSecuritiesCurrent", "AvailableForSaleSecuritiesDebtSecuritiesCurrent", "ShortTermInvestments"]),
    "accounts_receivable":("BS", ["AccountsReceivableNetCurrent"]),
    "inventory":          ("BS", ["InventoryNet"]),
    "current_assets":     ("BS", ["AssetsCurrent"]),
    "total_assets":       ("BS", ["Assets"]),
    "current_liabilities":("BS", ["LiabilitiesCurrent"]),
    "total_liabilities":  ("BS", ["Liabilities"]),
    "long_term_debt":     ("BS", ["LongTermDebtNoncurrent", "LongTermDebt"]),
    "total_equity":       ("BS", ["StockholdersEquity"]),
    "liab_and_equity":    ("BS", ["LiabilitiesAndStockholdersEquity"]),
    "goodwill":           ("BS", ["Goodwill"]),
    # Cash flow
    "operating_cash_flow":("CF", ["NetCashProvidedByUsedInOperatingActivities"]),
    "capex":              ("CF", ["PaymentsToAcquirePropertyPlantAndEquipment"]),
    "depreciation":       ("CF", ["Depreciation", "DepreciationDepletionAndAmortization", "DepreciationAmortizationAndAccretionNet"]),
    "stock_comp":         ("CF", ["ShareBasedCompensation", "AllocatedShareBasedCompensationExpense"]),
    "buybacks":           ("CF", ["PaymentsForRepurchaseOfCommonStock"]),
    "acquisitions":       ("CF", ["PaymentsToAcquireBusinessesNetOfCashAcquired"]),
}

In [9]:
records, report = [], []
for metric, (statement, candidates) in METRICS.items():
    sub = df[df["concept"].isin(candidates)].copy()
    if sub.empty:
        report.append((metric, "missing", ""))
        continue
    sub["priority"] = sub["concept"].map({c: i for i, c in enumerate(candidates)})
    sub = sub.sort_values("priority").drop_duplicates("fiscal_year", keep="first")
    sub["metric"], sub["statement"] = metric, statement
    records.append(sub)
    report.append((metric, "ok", ", ".join(sub["concept"].unique())))

fin_long = pd.concat(records, ignore_index=True)
report = pd.DataFrame(report, columns=["metric", "status", "tag_used"])
print(report["status"].value_counts().to_dict())
report[report["status"] != "ok"]

{'ok': 28}


,metric,status,tag_used


### 4.1 Derive missing metrics from accounting identities

Where a total is not reported, it is computed (`total liabilities = liabilities & equity − equity`; `gross profit = revenue − cost of revenue`) and flagged with `concept = "derived"` so it is never confused with a reported figure.

In [10]:
wide = fin_long.pivot_table(index="fiscal_year", columns="metric", values="val")

def fill_derived(metric, formula_values, statement):
    global fin_long
    if metric not in wide:
        wide[metric] = float("nan")
    missing = wide[metric].isna() & formula_values.notna()
    if missing.any():
        wide.loc[missing, metric] = formula_values[missing]
        add = pd.DataFrame({"fiscal_year": wide.index[missing], "metric": metric,
                            "val": formula_values[missing].values, "concept": "derived",
                            "statement": statement})
        fin_long = pd.concat([fin_long, add], ignore_index=True)
    print(f"{metric}: {missing.sum()} year(s) derived")

fill_derived("total_liabilities", wide["liab_and_equity"] - wide["total_equity"], "BS")
fill_derived("gross_profit", wide["revenue"] - wide["cost_of_revenue"], "IS")

total_liabilities: 0 year(s) derived
gross_profit: 0 year(s) derived


## 5. Validate

1. Balance sheet identity: assets = liabilities + equity
2. Gross profit = revenue − cost of revenue
3. Reconcile with reported revenue: FY2023 $663.1M, FY2024 $763.6M, FY2025 $785.0M

In [11]:
check = pd.DataFrame({
    "assets - liabilities - equity": wide["total_assets"] - wide["total_liabilities"] - wide["total_equity"],
    "gross profit - (revenue - COGS)": wide["gross_profit"] - (wide["revenue"] - wide["cost_of_revenue"]),
})
assert check.abs().max().max() < 1, "Accounting identity check failed"
print("Both identities hold for every year")
(wide[["revenue", "gross_profit", "operating_income", "net_income", "operating_cash_flow"]].tail(10) / 1e6).round(1)

Both identities hold for every year


metric,revenue,gross_profit,operating_income,net_income,operating_cash_flow
fiscal_year,,,,,
2016,383.90,102.70,-47.90,-6.60,NaN
2017,548.40,215.60,46.30,40.90,86.30
2018,529.70,210.30,36.10,104.00,68.70
2019,589.50,237.50,49.70,39.30,121.00
2020,693.60,287.90,83.80,78.50,169.30
2021,769.70,322.80,98.00,83.90,139.40
2022,747.90,296.00,54.90,50.70,131.80
2023,663.10,258.60,82.80,82.40,64.60
2024,763.60,307.90,64.80,69.60,117.50


## 6. Export

In [12]:
last_fy = int(wide.index.max())
keep_years = range(last_fy - 9, last_fy + 1)        # latest 10 fiscal years

out_long = (fin_long[fin_long["fiscal_year"].isin(keep_years)]
            [["fiscal_year", "statement", "metric", "val", "concept", "end", "filed", "accn"]]
            .rename(columns={"val": "value", "end": "period_end", "filed": "filed_date"})
            .sort_values(["statement", "metric", "fiscal_year"]))
out_long.insert(0, "ticker", "FORM")
out_long.to_csv(CLEAN_DIR / "formfactor_financials_long.csv", index=False)

out_wide = wide.loc[wide.index.isin(keep_years)].reindex(columns=list(METRICS.keys()))
out_wide.to_csv(CLEAN_DIR / "formfactor_financials_wide.csv")

pd.DataFrame([(m, s, " | ".join(c)) for m, (s, c) in METRICS.items()],
             columns=["metric", "statement", "xbrl_candidates"]).to_csv(CLEAN_DIR / "metric_dictionary.csv", index=False)

for p in sorted(CLEAN_DIR.glob("formfactor*.csv")) + [CLEAN_DIR / "metric_dictionary.csv"]:
    print(f"{p.name:35s} {len(pd.read_csv(p)):4d} rows")

formfactor_financials_long.csv       278 rows
formfactor_financials_wide.csv        10 rows
metric_dictionary.csv                 28 rows


## Data notes

- All 28 metrics were found in FormFactor's filings; no values had to be derived from identities, and both accounting identities hold in every year.
- FY2016 is affected by the mid-year acquisition of Cascade Microtech (partial-year revenue and deal costs) and is excluded from trend analysis in later notebooks.
- Raw API responses are cached and not committed to the repository; the cleaned CSVs in `data/clean/` are the inputs to notebook 02.